K-mer Approach

In [ ]:
## This scripts finds overlapping sequences in datasets
## I saved it in "/fh/fast/hill_g/Albert/Collaboration-Microbiome/Script/find_shared_regions.py"
# Builds a k-mer (default 35-mer) index on the reference sequence.
# For each other sequence, collects its k-mers and counts support for each reference k-mer.
# Keeps only k-mers present in all sequences (or at least N if you pass --require-at-least N).
# On the reference, stitches adjacent shared k-mer windows into maximal continuous regions ≥k, and outputs those regions with 1-based start/end.

# Basic: shared regions present in ALL sequences, k=35, reference = first sequence found
# python find_shared_regions.py /path/to/fasta_dir -o shared.tsv
# Require presence in at least 10 sequences (not necessarily all)
# python find_shared_regions.py /path/to/fasta_dir -k 40 --require-at-least 10 -o shared_atleast10.tsv
# Choose a specific reference by name substring (e.g., pick a particular GCA/GCF record)
# python find_shared_regions.py /path/to/fasta_dir --ref-name GCA_000020225.1 -o shared_GCA000020225.tsv

import argparse
import os
from pathlib import Path

def read_fasta(path):
    """Yield (header, sequence) for all records in a FASTA file."""
    header = None
    chunks = []
    with open(path, 'r') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if header is not None:
                    yield header, ''.join(chunks).upper().replace('U', 'T')
                header = line[1:].strip()
                chunks = []
            else:
                chunks.append(line)
        if header is not None:
            yield header, ''.join(chunks).upper().replace('U', 'T')

def collect_sequences(fasta_dir):
    """Return list of (name, seq) across all FASTA files in a directory."""
    seqs = []
    for p in sorted(Path(fasta_dir).glob("*.fasta")):
        for hdr, seq in read_fasta(p):
            # Make a short, unique name combining file stem and record id
            rid = hdr.split()[0]
            seqs.append((f"{p.stem}|{rid}", seq))
    return seqs

def kmers_positions(seq, k):
    """Return dict: kmer -> list of start positions (0-based) in seq."""
    d = {}
    n = len(seq)
    if n < k:
        return d
    for i in range(n - k + 1):
        kmer = seq[i:i+k]
        if 'N' in kmer:
            continue
        d.setdefault(kmer, []).append(i)
    return d

def main():
    ap = argparse.ArgumentParser(
        description="Find shared continuous sequences (exact matches) of length >= k across FASTA files."
    )
    ap.add_argument("fasta_dir", help="Directory containing .fasta files")
    ap.add_argument("-k", "--kmer", type=int, default=35, help="Minimum length (k) for shared sequences (default: 35)")
    ap.add_argument("--require-at-least", type=int, default=0,
                    help="Require presence in at least this many sequences (default: 0 -> all sequences)")
    ap.add_argument("--ref-name", type=str, default=None,
                    help="Name (substring match) of reference sequence to use for coordinates")
    ap.add_argument("-o", "--out", type=str, default="shared_regions.tsv",
                    help="Output TSV (sequence, start, end) with 1-based coordinates relative to reference")
    args = ap.parse_args()

    seqs = collect_sequences(args.fasta_dir)
    if not seqs:
        raise SystemExit("No FASTA records found.")

    # Choose reference
    if args.ref_name:
        ref_candidates = [t for t in seqs if args.ref_name in t[0]]
        if not ref_candidates:
            raise SystemExit(f"--ref-name '{args.ref_name}' not found among sequence names.")
        ref_name, ref_seq = ref_candidates[0]
    else:
        ref_name, ref_seq = seqs[0]

    # Build k-mer index for reference
    k = args.kmer
    ref_index = kmers_positions(ref_seq, k)
    if not ref_index:
        raise SystemExit(f"Reference sequence '{ref_name}' shorter than k={k} or no valid k-mers.")

    # Determine how many sequences must contain each k-mer
    if args.require_at_least and args.require_at_least > 0:
        required = args.require_at_least
    else:
        required = len(seqs)  # require presence in all sequences

    # For each non-reference sequence, build the set of its k-mers
    # Count in how many sequences each reference k-mer appears
    kmer_support = {kmer: 1 for kmer in ref_index.keys()}  # reference counts as 1
    for name, seq in seqs:
        if name == ref_name:
            continue
        s_kmers = set(kmers_positions(seq, k).keys())
        for km in kmer_support.keys():
            if km in s_kmers:
                kmer_support[km] += 1

    # Keep only k-mers that reach required support
    shared_kmers = {km for km, c in kmer_support.items() if c >= required}
    if not shared_kmers:
        # Relax message in case user wants at-least threshold
        need = f"at least {required}" if required < len(seqs) else "all"
        raise SystemExit(f"No shared k-mers (k={k}) found across {need} sequences.")

    # On the reference, mark positions where the k-mer window is shared
    n = len(ref_seq)
    is_shared = [False] * n
    # For each shared k-mer, mark its window(s) in the reference
    for km in shared_kmers:
        for pos in ref_index[km]:
            for i in range(pos, pos + k):
                is_shared[i] = True

    # Extract maximal continuous regions where every k-mer window is shared.
    # A region qualifies if its length >= k AND every internal k-mer is shared.
    regions = []
    i = 0
    while i < n:
        # Skip positions that are not part of any shared k-mer span
        while i < n and not is_shared[i]:
            i += 1
        if i >= n:
            break
        # Grow region while positions are marked shared
        j = i
        while j < n and is_shared[j]:
            j += 1
        # Candidate region is [i, j) (0-based, end-exclusive)
        if j - i >= k:
            # Additional check: ensure every k-mer fully inside [i, j) is shared
            # This is implicitly true by our marking, but we can be strict:
            ok = True
            for s in range(i, j - k + 1):
                if ref_seq[s:s+k] not in shared_kmers:
                    ok = False
                    break
            if ok:
                regions.append((i, j))  # 0-based
        i = j

    # Merge adjacent/overlapping regions that are contiguous (optional clean-up)
    merged = []
    for start, end in regions:
        if not merged or start > merged[-1][1]:
            merged.append([start, end])
        else:
            merged[-1][1] = max(merged[-1][1], end)

    # Write output
    with open(args.out, "w") as out:
        # 3 columns: sequence, start, end (1-based positions relative to reference)
        for s, e in merged:
            subseq = ref_seq[s:e]
            out.write(f"{subseq}\t{s+1}\t{e}\n")

    print(f"Reference: {ref_name}")
    print(f"Sequences loaded: {len(seqs)}")
    print(f"k = {k}, required presence = {required} sequences")
    print(f"Shared regions found: {len(merged)}")
    print(f"Wrote: {args.out}")

if __name__ == "__main__":
    main()


In [5]:
## How read_fasta works

def read_fasta(path):
    """
    Yield (header, sequence) tuples from a FASTA file.

    Each sequence is returned as uppercase DNA (U -> T).
    Example usage:
        for hdr, seq in read_fasta("example.fasta"):
            print(hdr, len(seq))
    """
    header = None
    chunks = []
    with open(path, "r") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(chunks).upper().replace("U", "T")
                header = line[1:].strip()
                chunks = []
            else:
                chunks.append(line)
        if header is not None:
            yield header, "".join(chunks).upper().replace("U", "T")
            
fasta_path = "/fh/fast/hill_g/Albert/Collaboration-Microbiome/NCBI/Akkermansia_muciniphila/16S_rRNA/GCA_000020225.1.fasta"

for header, seq in read_fasta(fasta_path):
    print(f"Header: {header}")
    print(f"Sequence length: {len(seq)}")
    print(f"First 60 bases: {seq[:60]}...")
    print("-" * 80)

Header: GCA_000020225.1|CP001071.1:320473-321977(-)|16S_rRNA_1
Sequence length: 1505
First 60 bases: AGAGTTTGATTCTGGCTCAGAACGAACGCTGGCGGCGTGGATAAGACATGCAAGTCGAAC...
--------------------------------------------------------------------------------
Header: GCA_000020225.1|CP001071.1:989916-991420(+)|16S_rRNA_2
Sequence length: 1505
First 60 bases: AGAGTTTGATTCTGGCTCAGAACGAACGCTGGCGGCGTGGATAAGACATGCAAGTCGAAC...
--------------------------------------------------------------------------------
Header: GCA_000020225.1|CP001071.1:1459761-1461265(+)|16S_rRNA_3
Sequence length: 1505
First 60 bases: AGAGTTTGATTCTGGCTCAGAACGAACGCTGGCGGCGTGGATAAGACATGCAAGTCGAAC...
--------------------------------------------------------------------------------


In [19]:
## How collect_sequences work

from pathlib import Path

def collect_sequences(fasta_dir):
    """Return list of (name, seq) across all FASTA files in a directory."""
    seqs = []
    for p in sorted(Path(fasta_dir).glob("*.fasta")):
        for hdr, seq in read_fasta(p):
            # Make a short, unique name combining file stem and record id
            rid = hdr.split()[0]
            seqs.append((f"{p.stem}|{rid}", seq))
    return seqs

fasta_path = "/fh/fast/hill_g/Albert/Collaboration-Microbiome/NCBI/Akkermansia_muciniphila/16S_rRNA/"
seqs = collect_sequences(fasta_path)
seqs[4]

('GCA_001688765.2|GCA_001688765.2|CP015409.2:1960463-1961978(-)|16S_rRNA_2',
 'ATGGAGAGTTTGATTCTGGCTCAGAACGAACGCTGGCGGCGTGGATAAGACATGCAAGTCGAACGGGAGAATTGCTAGCTTGCTAATAATTCTCTAGTGGCGCACGGGTGAGTAACACGTGAGTAACCTGCCCCCGAGAGCGGGATAGCCCTGGGAAACTGGGATTAATACCGCATAGTATCGAAAGATTAAAGCAGCAATGCGCTTGGGGATGGGCTCGCGGCCTATTAGTTAGTTGGTGAGGTAACGGCTCACCAAGGCGATGACGGGTAGCCGGTCTGAGAGGATGTCCGGCCACACTGGAACTGAGACACGGTCCAGACACCTACGGGTGGCAGCAGTCGAGAATCATTCACAATGGGGGAAACCCTGATGGTGCGACGCCGCGTGGGGGAATGAAGGTCTTCGGATTGTAAACCCCTGTCATGTGGGAGCAAATTAAAAAGATAGTACCACAAGAGGAAGAGACGGCTAACTCTGTGCCAGCAGCCGCGGTAATACAGAGGTCTCAAGCGTTGTTCGGAATCACTGGGCGTAAAGCGTGCGTAGGCTGTTTCGTAAGTCGTGTGTGAAAGGCGCGGGCTCAACCCGCGGACGGCACATGATACTGCGAGACTAGAGTAATGGAGGGGGAACCGGAATTCTCGGTGTAGCAGTGAAATGCGTAGATATCGAGAGGAACACTCGTGGCGAAGGCGGGTTCCTGGACATTAACTGACGCTGAGGCACGAAGGCCAGGGGAGCGAAAGGGATTAGATACCCCTGTAGTCCTGGCAGTAAACGGTGCACGCTTGGTGTGCGGGGAATCGACCCCCTGCGTGCCGGAGCTAACGCGTTAAGCGTGCCGCCTGGGGAGTACGGTCGCAAGATTAAAACTCAAAGAAATTGACGGGGACCCGCACAAGCGGTGGAGTATGTGGC

In [23]:
## How kmers_positions work

def kmers_positions(seq, k):
    """Return dict: kmer -> list of start positions (0-based) in seq."""
    d = {}
    n = len(seq)
    if n < k:
        return d
    for i in range(n - k + 1):
        kmer = seq[i:i+k]
        if 'N' in kmer:
            continue
        d.setdefault(kmer, []).append(i)
    return d

seq = "ATGCATGCATGCATGCGAT"
k = 8
print(kmers_positions(seq, k))


{'ATGCATGC': [0, 4, 8], 'TGCATGCA': [1, 5], 'GCATGCAT': [2, 6], 'CATGCATG': [3, 7], 'TGCATGCG': [9], 'GCATGCGA': [10], 'CATGCGAT': [11]}


In [ ]:
#Allows input of type strain
#!/usr/bin/env python3
import argparse
import math
from pathlib import Path
from typing import Optional


def read_fasta(path: Path):
    """Yield (header, sequence) for all records in a FASTA file."""
    header = None
    chunks = []
    with path.open("r") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(chunks).upper().replace("U", "T")
                header = line[1:].strip()
                chunks = []
            else:
                chunks.append(line)
        if header is not None:
            yield header, "".join(chunks).upper().replace("U", "T")


def kmers_positions(seq: str, k: int):
    """Return dict: kmer -> list of start positions (0-based) in seq. Skips kmers containing 'N'."""
    d = {}
    n = len(seq)
    if n < k:
        return d
    for i in range(n - k + 1):
        kmer = seq[i:i+k]
        if 'N' in kmer:
            continue
        d.setdefault(kmer, []).append(i)
    return d


def kmers_set(seq: str, k: int):
    """Return set of k-mers (no positions), skipping kmers containing 'N'."""
    out = set()
    n = len(seq)
    if n < k:
        return out
    for i in range(n - k + 1):
        kmer = seq[i:i+k]
        if 'N' in kmer:
            continue
        out.add(kmer)
    return out


def load_records_and_strain_kmer_sets(fasta_dir: Path, k: int):
    """
    Returns:
      records: list of (strain_id, record_name, sequence) across all .fasta files
      strain_kmers: dict[strain_id] -> set of kmers (union across that file's records)
    """
    records = []
    strain_kmers = {}
    for p in sorted(fasta_dir.glob("*.fasta")):
        strain_id = p.stem  # each file is one strain
        union = set()
        for hdr, seq in read_fasta(p):
            record_name = f"{strain_id}|{hdr.split()[0]}"
            records.append((strain_id, record_name, seq))
            union |= kmers_set(seq, k)
        strain_kmers[strain_id] = union
    return records, strain_kmers


def compute_required(n_strains: int, min_fraction: Optional[float], require_at_least: int):
    """
    Decide how many STRAINS must have the k-mer.
    Priority: --min-fraction (if provided) > --require-at-least (>0) > default (= all strains).
    """
    if min_fraction is not None:
        f = float(min_fraction)
        if f < 0 or f > 1.0:
            raise SystemExit(f"--min-fraction must be in [0,1], got {f}")
        if f == 0.0:
            return 1
        return max(1, math.ceil(f * n_strains))
    if require_at_least and require_at_least > 0:
        return require_at_least
    return n_strains


def main():
    ap = argparse.ArgumentParser(
        description="Find shared continuous sequences (exact matches) of length >= k across FASTA files, "
                    "counting support by number of strains (files), not total records."
    )
    ap.add_argument("fasta_dir", help="Directory containing .fasta files (each file = one strain)")
    ap.add_argument("-k", "--kmer", type=int, default=35, help="Minimum k (default: 35)")
    ap.add_argument("--require-at-least", type=int, default=0,
                    help="Require presence in at least this many strains (default: 0 -> all strains). "
                         "Ignored if --min-fraction is set.")
    ap.add_argument("--min-fraction", type=float, default=None,
                    help="Fraction of strains that must contain a k-mer (0..1]. "
                         "0 => keep k-mers seen in >=1 strain. Takes precedence over --require-at-least.")
    ap.add_argument("--ref-name", type=str, default=None,
                    help="Substring to select a specific reference record for coordinates "
                         "(searches across all records' names). Defaults to the first record.")
    ap.add_argument("--ref-file", type=str, default=None,
                    help="Filename of the reference strain (e.g. GCF_029910175.1.fasta). "
                         "Overrides --ref-name if provided.")
    ap.add_argument("-o", "--out", type=str, default="shared_regions.tsv",
                    help="Output TSV with columns: sequence, start, end (1-based on reference)")
    args = ap.parse_args()

    fasta_dir = Path(args.fasta_dir)
    if not fasta_dir.is_dir():
        raise SystemExit(f"Not a directory: {fasta_dir}")

    # Preload records and per-strain k-mer unions
    k = args.kmer
    records, strain_kmers = load_records_and_strain_kmer_sets(fasta_dir, k)
    if not records:
        raise SystemExit("No FASTA records found.")

    # Pick reference record
    if args.ref_file:
        ref_path = fasta_dir / args.ref_file
        if not ref_path.exists():
            raise SystemExit(f"--ref-file '{args.ref_file}' not found in directory.")
        ref_records = list(read_fasta(ref_path))
        if not ref_records:
            raise SystemExit(f"No FASTA records found in {args.ref_file}.")
        ref_strain = ref_path.stem
        ref_name, ref_seq = ref_records[0]
    elif args.ref_name:
        ref_candidates = [(sid, rname, seq) for (sid, rname, seq) in records if args.ref_name in rname]
        if not ref_candidates:
            raise SystemExit(f"--ref-name '{args.ref_name}' not found among record names.")
        ref_strain, ref_name, ref_seq = ref_candidates[0]
    else:
        ref_strain, ref_name, ref_seq = records[0]

    # Build reference k-mer index
    ref_index = kmers_positions(ref_seq, k)
    if not ref_index:
        raise SystemExit(f"Reference record '{ref_name}' shorter than k={k} or no valid k-mers.")

    # Determine strain-based support threshold
    n_strains = len(strain_kmers)
    required = compute_required(n_strains, args.min_fraction, args.require_at_least)

    # Count in how many STRAINS each reference k-mer appears
    kmer_support = {}
    ref_union = strain_kmers.get(ref_strain, set())
    for km in ref_index.keys():
        kmer_support[km] = 1 if km in ref_union else 0

    for sid, union in strain_kmers.items():
        if sid == ref_strain:
            continue
        for km in kmer_support.keys():
            if km in union:
                kmer_support[km] += 1

    # Keep only reference kmers supported by enough STRAINS
    shared_kmers = {km for km, c in kmer_support.items() if c >= required}
    if not shared_kmers:
        need = (f"ceil({args.min_fraction}*{n_strains})={required}" if args.min_fraction is not None
                else (f"{required}" if args.require_at_least > 0 else "all strains"))
        raise SystemExit(f"No shared k-mers (k={k}) reached threshold across {need}.")

    # Mark shared positions on the reference record
    n = len(ref_seq)
    is_shared = [False] * n
    for km in shared_kmers:
        for pos in ref_index[km]:
            for i in range(pos, pos + k):
                is_shared[i] = True

    # Extract continuous regions
    regions = []
    i = 0
    while i < n:
        while i < n and not is_shared[i]:
            i += 1
        if i >= n:
            break
        j = i
        while j < n and is_shared[j]:
            j += 1
        if j - i >= k:
            ok = True
            for s in range(i, j - k + 1):
                if ref_seq[s:s+k] not in shared_kmers:
                    ok = False
                    break
            if ok:
                regions.append((i, j))
        i = j

    # Merge contiguous/overlapping intervals
    merged = []
    for start, end in regions:
        if not merged or start > merged[-1][1]:
            merged.append([start, end])
        else:
            merged[-1][1] = max(merged[-1][1], end)

    # Summary stats
    total_bp = sum(end - start for start, end in merged)
    total_records = len(records)
    total_strains = len(strain_kmers)
    used_fraction = args.min_fraction if args.min_fraction is not None else "None"

    # Write output with header block
    with open(args.out, "w") as out:
        out.write(f">Reference_record: {ref_name}\n")
        out.write(f"# Reference_file: {args.ref_file if args.ref_file else 'Auto-selected'}\n")
        out.write(f"# Strains_analyzed: {total_strains}\n")
        out.write(f"# Total_unique_sequences: {total_records}\n")
        out.write(f"# min_fraction_used: {used_fraction}\n")
        out.write(f"# k={k}, Required_support={required}, Total_shared_regions={len(merged)}, Total_bp_matched={total_bp}\n")
        out.write("sequence\tstart\tend\n")
        for s, e in merged:
            subseq = ref_seq[s:e]
            out.write(f"{subseq}\t{s+1}\t{e}\n")

    # Console summary
    print(f"Reference record: {ref_name} (strain={ref_strain})")
    if args.ref_file:
        print(f"Reference file: {args.ref_file}")
    print(f"Strains (files): {total_strains}")
    print(f"Total unique sequences analyzed: {total_records}")
    print(f"min_fraction used: {used_fraction}")
    if args.min_fraction is not None:
        print(f"k = {k}, strain support >= ceil({args.min_fraction} * {total_strains}) = {required}")
    else:
        th = required if args.require_at_least > 0 else total_strains
        print(f"k = {k}, strain support >= {th}")
    print(f"Shared regions: {len(merged)}")
    print(f"Total basepairs matched: {total_bp}")
    print(f"Wrote: {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
# Using first reference file (not type strain)
#!/usr/bin/env python3
import argparse
import math
from pathlib import Path
from typing import Optional


def read_fasta(path: Path):
    """Yield (header, sequence) for all records in a FASTA file."""
    header = None
    chunks = []
    with path.open("r") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(chunks).upper().replace("U", "T")
                header = line[1:].strip()
                chunks = []
            else:
                chunks.append(line)
        if header is not None:
            yield header, "".join(chunks).upper().replace("U", "T")

def kmers_positions(seq: str, k: int):
    """Return dict: kmer -> list of start positions (0-based) in seq. Skips kmers containing 'N'."""
    d = {}
    n = len(seq)
    if n < k:
        return d
    for i in range(n - k + 1):
        kmer = seq[i:i+k]
        if 'N' in kmer:
            continue
        d.setdefault(kmer, []).append(i)
    return d

def kmers_set(seq: str, k: int):
    """Return set of k-mers (no positions), skipping kmers containing 'N'."""
    out = set()
    n = len(seq)
    if n < k:
        return out
    for i in range(n - k + 1):
        kmer = seq[i:i+k]
        if 'N' in kmer:
            continue
        out.add(kmer)
    return out

def load_records_and_strain_kmer_sets(fasta_dir: Path, k: int):
    """
    Returns:
      records: list of (strain_id, record_name, sequence) across all .fasta files
      strain_kmers: dict[strain_id] -> set of kmers (union across that file's records)
    """
    records = []
    strain_kmers = {}
    for p in sorted(fasta_dir.glob("*.fasta")):
        strain_id = p.stem  # each file is one strain
        union = set()
        for hdr, seq in read_fasta(p):
            record_name = f"{strain_id}|{hdr.split()[0]}"
            records.append((strain_id, record_name, seq))
            union |= kmers_set(seq, k)
        strain_kmers[strain_id] = union
    return records, strain_kmers

def compute_required(n_strains: int, min_fraction: Optional[float], require_at_least: int):
    """
    Decide how many STRAINS must have the k-mer.
    Priority: --min-fraction (if provided) > --require-at-least (>0) > default (= all strains).
    """
    if min_fraction is not None:
        f = float(min_fraction)
        if f < 0 or f > 1.0:
            raise SystemExit(f"--min-fraction must be in [0,1], got {f}")
        if f == 0.0:
            return 1
        return max(1, math.ceil(f * n_strains))
    if require_at_least and require_at_least > 0:
        return require_at_least
    return n_strains

def main():
    ap = argparse.ArgumentParser(
        description="Find shared continuous sequences (exact matches) of length >= k across FASTA files, "
                    "counting support by number of strains (files), not total records."
    )
    ap.add_argument("fasta_dir", help="Directory containing .fasta files (each file = one strain)")
    ap.add_argument("-k", "--kmer", type=int, default=35, help="Minimum k (default: 35)")
    ap.add_argument("--require-at-least", type=int, default=0,
                    help="Require presence in at least this many strains (default: 0 -> all strains). "
                         "Ignored if --min-fraction is set.")
    ap.add_argument("--min-fraction", type=float, default=None,
                    help="Fraction of strains that must contain a k-mer (0..1]. "
                         "0 => keep k-mers seen in >=1 strain. Takes precedence over --require-at-least.")
    ap.add_argument("--ref-name", type=str, default=None,
                    help="Substring to select a specific reference record for coordinates "
                         "(searches across all records' names). Defaults to the first record.")
    ap.add_argument("-o", "--out", type=str, default="shared_regions.tsv",
                    help="Output TSV with columns: sequence, start, end (1-based on reference)")
    args = ap.parse_args()

    fasta_dir = Path(args.fasta_dir)
    if not fasta_dir.is_dir():
        raise SystemExit(f"Not a directory: {fasta_dir}")

    # Preload records and per-strain k-mer unions
    k = args.kmer
    records, strain_kmers = load_records_and_strain_kmer_sets(fasta_dir, k)
    if not records:
        raise SystemExit("No FASTA records found.")

    # Pick reference record (for coordinates only)
    if args.ref_name:
        ref_candidates = [(sid, rname, seq) for (sid, rname, seq) in records if args.ref_name in rname]
        if not ref_candidates:
            raise SystemExit(f"--ref-name '{args.ref_name}' not found among record names.")
        ref_strain, ref_name, ref_seq = ref_candidates[0]
    else:
        ref_strain, ref_name, ref_seq = records[0]

    # Build reference k-mer index (positions on that single record)
    ref_index = kmers_positions(ref_seq, k)
    if not ref_index:
        raise SystemExit(f"Reference record '{ref_name}' shorter than k={k} or no valid k-mers.")

    # Determine strain-based support threshold
    n_strains = len(strain_kmers)
    required = compute_required(n_strains, args.min_fraction, args.require_at_least)

    # Count in how many STRAINS each reference k-mer appears.
    # Initialize with 1 if the reference strain's union contains the k-mer, else 0 (should be 1).
    kmer_support = {}
    ref_union = strain_kmers.get(ref_strain, set())
    for km in ref_index.keys():
        kmer_support[km] = 1 if km in ref_union else 0  # robust, though ref kmers should be in ref_union

    # For every OTHER strain, increment if its union contains the ref k-mer
    for sid, union in strain_kmers.items():
        if sid == ref_strain:
            continue
        for km in kmer_support.keys():
            if km in union:
                kmer_support[km] += 1

    # Keep only reference kmers supported by enough STRAINS
    shared_kmers = {km for km, c in kmer_support.items() if c >= required}
    if not shared_kmers:
        need = (f"ceil({args.min_fraction}*{n_strains})={required}" if args.min_fraction is not None
                else (f"{required}" if args.require_at_least > 0 else "all strains"))
        raise SystemExit(f"No shared k-mers (k={k}) reached threshold across {need}.")

    # Mark shared positions on the reference record
    n = len(ref_seq)
    is_shared = [False] * n
    for km in shared_kmers:
        for pos in ref_index[km]:
            for i in range(pos, pos + k):
                is_shared[i] = True

    # Extract maximal continuous regions (every internal k-mer must be shared)
    regions = []
    i = 0
    while i < n:
        while i < n and not is_shared[i]:
            i += 1
        if i >= n:
            break
        j = i
        while j < n and is_shared[j]:
            j += 1
        if j - i >= k:
            ok = True
            for s in range(i, j - k + 1):
                if ref_seq[s:s+k] not in shared_kmers:
                    ok = False
                    break
            if ok:
                regions.append((i, j))
        i = j

    # Merge contiguous/overlapping intervals
    merged = []
    for start, end in regions:
        if not merged or start > merged[-1][1]:
            merged.append([start, end])
        else:
            merged[-1][1] = max(merged[-1][1], end)

    # Compute summary stats
    total_bp = sum(end - start for start, end in merged)
    total_records = len(records)
    total_strains = len(strain_kmers)
    used_fraction = args.min_fraction if args.min_fraction is not None else "None"

    # Write output with header block
    with open(args.out, "w") as out:
        out.write(f">Reference_record: {ref_name}\n")
        out.write(f"# Strains_analyzed: {total_strains}\n")
        out.write(f"# Total_unique_sequences: {total_records}\n")
        out.write(f"# min_fraction_used: {used_fraction}\n")
        out.write(f"# k={k}, Required_support={required}, Total_shared_regions={len(merged)}, Total_bp_matched={total_bp}\n")
        out.write("sequence\tstart\tend\n")
        for s, e in merged:
            subseq = ref_seq[s:e]
            out.write(f"{subseq}\t{s+1}\t{e}\n")

    # Console summary
    print(f"Reference record: {ref_name} (strain={ref_strain})")
    print(f"Strains (files): {total_strains}")
    print(f"Total unique sequences analyzed: {total_records}")
    print(f"min_fraction used: {used_fraction}")
    if args.min_fraction is not None:
        print(f"k = {k}, strain support >= ceil({args.min_fraction} * {total_strains}) = {required}")
    else:
        th = required if args.require_at_least > 0 else total_strains
        print(f"k = {k}, strain support >= {th}")
    print(f"Shared regions: {len(merged)}")
    print(f"Total basepairs matched: {total_bp}")
    print(f"Wrote: {args.out}")
    
if __name__ == "__main__":
    main()